**Import packages and dependecies**

In [ ]:
%pip install -q -e ..

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import kagglehub

from tda.dimensionality_reduction import UMAP
from tda.matching import mapper_bin_and_cluster

### 1. Load in data inputs

The dataset below contains roughly 30,000 member records on various health conditions and risk factors, including:

- Demographics (e.g. age, gender)
- Chronic conditions (e.g. diabetes, hypertension)
- Biometric (e.g. glucose level, blood pressure)
- Medical risk factors (e.g. smoking, alcohol usage)
- Family medical history

For further information about the data, see [Healthcare Risk Factors Dataset](https://www.kaggle.com/datasets/abdallaahmed77/healthcare-risk-factors-dataset).

In [ ]:
# Download data fram Kaggle
path = kagglehub.dataset_download(
    "abdallaahmed77/healthcare-risk-factors-dataset"
)

health_risk_factors_df = pl.read_csv(f"{path}/dirty_v3_path.csv")

# Show data
health_risk_factors_df.limit(50).show()

**List of features fields.**

In [ ]:
features_list = [
    "Age",
    "Gender",
    "Medical Condition",
    "Glucose",
    "Blood Pressure",
    "BMI",
    "Oxygen Saturation",
    "LengthOfStay",
    "Cholesterol",
    "Triglycerides",
    "HbA1c",
    "Smoking",
    "Alcohol",
    "Physical Activity",
    "Diet Score",
    "Family History",
    "Stress Level",
    "Sleep Hours",
]

health_risk_factors_df = health_risk_factors_df.select(features_list)
print(f"Total # of features: {len(features_list)}")

### 2. Apply data preprocessing

**Remove columns if they have high missingness rate.**

In [ ]:
# Initialize parameters
tot_n_rows = health_risk_factors_df.shape[0]
missing_threshold = (
    0.4  # Columns w/ missing rate less than or equal to threshold are kept
)

# Calculate the percent of missing in each column
missing_count_df = health_risk_factors_df.null_count() / tot_n_rows
missing_count_df = missing_count_df.unpivot(
    on=features_list, variable_name="Variable Name", value_name="p_missing"
)

# Remove columns from data
columns_failed_threshold_list = (
    missing_count_df.filter(pl.col("p_missing") > missing_threshold)
    .select("Variable Name")
    .to_series()
)
preprocessed_risk_factors_df = health_risk_factors_df.drop(*columns_failed_threshold_list)
features_list = [c for c in features_list if c not in columns_failed_threshold_list]
print(
    f"Total # of columns removed due to high missing rate: {len(columns_failed_threshold_list)}"
)

**Apply mean imputation to numeric fields**

In [ ]:
preprocessed_risk_factors_df = preprocessed_risk_factors_df.with_columns(
    cs.numeric().fill_null(cs.numeric().mean())
)

preprocessed_risk_factors_df.show()

**Apply one-hot encoding to categorical fields**

In [ ]:
# List of all string type columns
string_dtype_list = [
    name
    for name, dtype in preprocessed_risk_factors_df.select(features_list).schema.items()
    if dtype == pl.String
]

# Apply one-hot encoding
preprocessed_risk_factors_df = preprocessed_risk_factors_df.to_dummies(string_dtype_list)
ohe_vars_list = [col for col in preprocessed_risk_factors_df.columns for s in string_dtype_list if col.startswith(s)] 
preprocessed_risk_factors_df.select(ohe_vars_list).show()



**Convert to float data type**

In [ ]:
preprocessed_risk_factors_df = preprocessed_risk_factors_df.select(pl.all().cast(pl.Float64))
preprocessed_risk_factors_df.show()

### 3. Perform topological matching using UMAP

**Apply dimensionality-reduction using UMAP**

We begin by performing dimensionality reduction on our data, keeping only the first 2 components. We apply UMAP in this example as our method of dimensionality reduction as it allows us to capture non-linearity in our lower dimensional space if needed.

In [ ]:
# Number of neighbors each data point should have
n_neighbors = 15
# Number of components or dimensions to reduce the set of all features down to
n_components = 2
# Number of iterations or epochs to use for gradient descent algorithm
n_epochs = 50
# Learning rate for gradient descent
lr = 0.1

# Run UMAP
mapper = UMAP(n_neighbors=n_neighbors, n_components=n_components, n_epochs=n_epochs, lr=lr)
embedding = mapper.fit_transform(preprocessed_risk_factors_df)

embedding.show()

**Bin and cluster data based on UMAP components**

We will now slice our UMAP component dimensions into multiple bins or intervals and perform clustering within each bin. It is important to note that the clustering is done using the raw, uncompressed, high-dimensional feature set rather than the dimensionally-reduced UMAP components; the UMAP components are only used to assign data points to bins. Because UMAP and, in general, any dimensionality reduction methods always introduces some amount of distortion or loss of information, it is better to cluster based on the raw features directly whenever possible. 

The bins in this situation mainly serve to guide local neighborhood division. We could, in theory, have skipped the binning and performed clustering immediately using the UMAP components, but standard clustering algorithms tend to create hard boundaries, assigning each data point to a single discrete cluster and as a consequence, possibly destroying any continuous topological structures like loops that may have existed in our data. In our binning process below, we actually allow each bin to overlap with one another so that a single data point can belong to multiple adjacent bins and be assigned to more than one cluster. This overlap connects local clusters into a simplical complex and maintains continuous topological shapes that traditional, hard clustering methods cannot.

In addition, global clustering algorithms often struggle when the data contains regions that vary drastically in density or structure from one another. By binning and localizing the data into smaller, bounded regions, we can better identify finer-grained sub-clusters within high-density data regions without missing coarse clusters in sparse data regions.


In [ ]:
# Number of bins to slice each UMAP dimension into
# e.g. if n_components = 2 and n_bins = 5, then each dimension is sliced into 5 bin, giving a total of 5*5 = 25 grid spaces
n_bins = 5
# The amount of overlap between UMAP bins
overlap = 5.0
# The epsilon value to use for DBSCAN (density-based spatial clustering algorithm)
# epsiolon represents the maximum distance between two samples for one to be considered in the neighborhood of the other
eps = 5.0
# The minimum number of data points within each bins in order to apply clustering
min_samples = 50


# Apply binning and clustering
clusters_df = mapper_bin_and_cluster(
    df_features=preprocessed_risk_factors_df,
    df_umap=embedding,
    n_bins=n_bins,
    overlap=overlap,
    eps=eps,
    min_samples=min_samples,
)

clusters_df.show()